# YOLO26n training - rocket parts

Trains the detector on v1 of our Roboflow dataset (nosecone, body tube, fin can, parachute).

Use a GPU runtime (T4 works). The Roboflow key goes in Colab secrets as `ROBOFLOW_API_KEY`.

In [ ]:
!pip install -q ultralytics roboflow
!nvidia-smi

In [ ]:
# runs go to drive so we don't lose them if colab disconnects
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/drone_yolo'

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
dataset = rf.workspace('anant-grover').project('real_drone_data').version(1).download('yolov8')
DATA_YAML = dataset.location + '/data.yaml'

## train

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
model.train(data=DATA_YAML, epochs=80, imgsz=640, batch=32,
            patience=20,  # stop early if val mAP stalls
            project=SAVE_DIR, name='run1', exist_ok=True)

## test set

Test split = photo sessions the model never saw during training (IMG_20xx-21xx and DJI_0230-0255).

In [ ]:
best = YOLO(SAVE_DIR + '/run1/weights/best.pt')
m = best.val(data=DATA_YAML, split='test', imgsz=640)

print('mAP50', round(m.box.map50, 3), ' mAP50-95', round(m.box.map, 3))
for i, name in best.names.items():
    p, r, ap50, ap = m.box.class_result(i)
    print(f'{name:10s} P={p:.2f} R={r:.2f} mAP50={ap50:.3f}')

In [ ]:
# quick look at a few test predictions
import glob
from IPython.display import Image, display

imgs = sorted(glob.glob(dataset.location + '/test/images/*'))[:8]
best.predict(imgs, conf=0.25, save=True, project=SAVE_DIR, name='test_preds', exist_ok=True)
for f in sorted(glob.glob(SAVE_DIR + '/test_preds/*'))[:8]:
    display(Image(filename=f, width=480))

## export

In [ ]:
# onnx -> gets compiled to .hef for the hailo-8 later
best.export(format='onnx', imgsz=640, opset=11, simplify=True)